# R5.1 — Which part of the detector carries the signal?

Reviewer 5, comment 1: compare prediction disagreement of (i) ordinary separately trained models,
(ii) contrastive Siamese models, and (iii) an explicit embedding-level measure.

Both variants share the published noisy labels and outer folds (rebuilt from the committed
prediction CSVs), the same seeded inner split, the same 200k training pairs per member, the same
initial weights, schedule and early stopping. They differ only in the objective:

| variant | objective |
|---|---|
| `siamese` | ours: CE on both branches + the contrastive loss `same·d² + (1-same)·[m-d]₊²` (`snd.training.contrastive`, the corrected form used for the CIFAR-10 runs) |
| `ce` | CE only (contrastive weight 0): an ordinary classifier ensemble on identical data |
| `ce_linear` | optional: backbone + plain linear head, CE only |

Each member saves, for the held-out outer fold, its softmax outputs and embeddings, and the
embeddings of its own training subset. The analysis then scores every held-out sample by
disagreement (the published detector), mean confidence in the observed label, and three
embedding-level scores computed inside each member (centroid margin, k-NN label disagreement,
across-member instability of the distance to the observed-label centroid; embedding spaces of
separately trained members are not aligned, so raw embedding variance is not meaningful).
It also reports the per-member misclassification rate on noisy vs clean samples (R4.3) and
paired-bootstrap CIs for every AUC difference.

**Budget (rough, refine from the first member's log):** CIFAR-10 ResNet-50 ~1.2–1.6 h per member
on a T4 with AMP, so one outer fold × 2 variants × 10 members ≈ 28 GPU-h ≈ 14 h on T4×2
(two commits). Fashion-MNIST ResNet-34 ≈ 0.3 h per member, so one outer fold ≈ 3–4 h on T4×2.

**Kaggle settings:** Accelerator *GPU T4 x2* (P100 also works, one worker), Internet *On*
(phone-verified account), Persistence *Files only*. Run with **Save Version -> Save & Run All
(Commit)** so it runs in the background for up to 12 h.

**Resuming:** jobs are saved as they finish. If the session ends with jobs left, open the
notebook, *Add Input -> Your Work -> this notebook* (the previous version's output), and
commit again: finished jobs are restored and skipped. Repeat until the launch cell prints
that nothing is left; the analysis cell then reports the final tables.

**Outputs** (in the version's Output tab): `r51_out/` raw per-job results, `results/` CSV
tables, `logs/` per-GPU training logs.

## 1. Configuration

In [ ]:
import json, os
CONFIG = dict(
    dataset='cifar10', noise=20,      # ('cifar10', 20|30|40) or ('fashionmnist', 20|30|40)
    preds_ref='main',                 # run whose predictions are the reference row: 'main' = the
                                      # CIFAR-10 re-run, '513d833' = the round-3 paper run (same
                                      # noise draw and folds, so training here is unaffected)
    code_ref='main',                  # branch providing the snd package
    outer_folds=[1],                  # outer fold(s) to score; fold 1 is unaffected by the
                                      # inner-model reuse in the CIFAR-10 re-run (folds 4, 5, 8)
    variants=['siamese', 'ce'],       # add 'ce_linear' if budget allows
    members=10, seed=0, amp=True,
    session_hours=11.5, member_hours=1.6,        # don't start a member that would cross 11.5 h
    save_checkpoints=True, loader_workers=2,
)
CONFIG.update(json.loads(os.environ.get('SND_CONFIG_OVERRIDES', '{}')))   # local smoke tests only
OUT_NAME, WORKER, PIP = 'r51_out', 'r51_worker.py', ['timm']
N_JOBS = len(CONFIG['outer_folds']) * CONFIG['members'] * len(CONFIG['variants'])

## 2. Setup (clone the repo, import kaggle/src)

In [ ]:
import json, os, subprocess, sys, time
SESSION_START = time.time()
os.chdir('/kaggle/working' if os.path.isdir('/kaggle/working') else os.getcwd())
subprocess.run('nvidia-smi -L', shell=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + PIP, check=False)
REPO = os.environ.get('SND_REPO') or '/tmp/snd/SiameseNoiseDetection'
if not os.path.isdir(os.path.join(REPO, '.git')):
    subprocess.run(['git', 'clone', '--quiet', 'https://github.com/stormaref/SiameseNoiseDetection.git',
                    REPO], check=True)
if not os.environ.get('SND_REPO'):                   # never move a local working copy
    subprocess.run(['git', '-C', REPO, 'checkout', '--quiet', CONFIG['code_ref']], check=True)
sys.path.insert(0, os.path.join(REPO, 'kaggle', 'src'))
import snd_kaggle as K
repo = K.setup_repo(CONFIG['code_ref'], REPO)
WORKER = os.path.join(REPO, 'kaggle', 'src', WORKER)
COMMIT = subprocess.run(['git', '-C', repo, 'log', '-1', '--format=%H %ad %s', '--date=iso'],
                        capture_output=True, text=True).stdout.strip()
print('code:', COMMIT)

## 3. Train (all GPUs, resumable)

In [ ]:
for train in (True, False):                     # download once, before the workers start
    K.base_dataset(CONFIG['dataset'], train)
CONFIG['out_dir'] = K.restore_previous_outputs(OUT_NAME)
CONFIG['session_start'] = SESSION_START
CONFIG['commit'] = COMMIT
with open(os.path.join(CONFIG['out_dir'], f'config_{int(SESSION_START)}.json'), 'w') as f:
    json.dump(CONFIG, f, indent=1)                  # provenance: config + commit per session
with open('config.json', 'w') as f:
    json.dump(CONFIG, f, indent=1)
codes = K.launch_workers(WORKER, 'config.json', poll=CONFIG.get('poll', 300))
print('worker exit codes:', codes)
done = len(K.JobQueue(CONFIG['out_dir']).summaries())
print(f'{done} of {N_JOBS} jobs finished' + ('' if done >= N_JOBS else
      ' -- commit a new version with this one attached as input to continue'))

## 4. Analysis

In [ ]:
import r51_analysis as A
table = K.load_label_table(K.extract_preds(repo, CONFIG['preds_ref'],
                                           K.PROTOCOL[(CONFIG['dataset'], CONFIG['noise'])]['preds']))
table, _ = K.smoke_subsample(table, CONFIG)
scores, thresholds, members, boot = A.report(CONFIG['out_dir'], table, 'results',
                                             min_members=CONFIG['members'])